In [1]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
print(f"Project root: {PROJECT_ROOT}")
DATA_DIR = PROJECT_ROOT / "data"

def read_json(file_path: Path | str) -> dict | list:
    data = None
    with open(file_path, "r") as f:
        data = json.load(f)
    return data

def read_jsonl(file_path: Path | str) -> list:
    res = []
    with open(file_path, "r") as f:
        for line in f:
            res.append(json.loads(line))
    return res

def write_json(data: dict | list, file_path: Path | str) -> None:
    with open(file_path, "w") as f:
        json.dump(data, f, indent=4)
        print(f"wrote to {file_path}")

Project root: /Users/matthewho/Documents/research/ctx_editor


In [2]:
# imported log folder at
# data/imported_lic_logs/sota_model_logs
LIC_LOGS_DIR = DATA_DIR / "imported_lic_logs/sota_model_logs"

# Load all JSONL log files into a single list of records
all_records = []
for jsonl_file in sorted(LIC_LOGS_DIR.rglob("*.jsonl")):
    records = read_jsonl(jsonl_file)
    all_records.extend(records)
    print(f"  {jsonl_file.relative_to(LIC_LOGS_DIR)}: {len(records)} records")

print(f"\nTotal records loaded: {len(all_records)}")

  actions/full/full_actions_gpt-5-chat.jsonl: 105 records
  actions/full/full_actions_gpt-5-mini.jsonl: 135 records
  actions/full/full_actions_gpt-5.1-chat.jsonl: 105 records
  actions/full/full_actions_gpt-5.1.jsonl: 105 records
  actions/full/full_actions_gpt-5.2-chat.jsonl: 345 records
  actions/sharded/sharded_actions_gpt-5-chat.jsonl: 104 records
  actions/sharded/sharded_actions_gpt-5-mini.jsonl: 133 records
  actions/sharded/sharded_actions_gpt-5.1-chat.jsonl: 207 records
  actions/sharded/sharded_actions_gpt-5.2-chat.jsonl: 340 records
  code/full/full_code_gpt-5-chat.jsonl: 100 records
  code/full/full_code_gpt-5-mini.jsonl: 130 records
  code/full/full_code_gpt-5.1-chat.jsonl: 100 records
  code/full/full_code_gpt-5.1.jsonl: 100 records
  code/full/full_code_gpt-5.2-chat.jsonl: 330 records
  code/sharded/sharded_code_gpt-5-chat.jsonl: 97 records
  code/sharded/sharded_code_gpt-5-mini.jsonl: 115 records
  code/sharded/sharded_code_gpt-5.1-chat.jsonl: 96 records
  code/sharded

In [3]:
# ── Step 1: Establish canonical correctness for each record ──
# Reconcile is_correct (bool|None) and score (float|None/NaN) into a single canonical value.

both_null_conv_ids = []
mismatch_conv_ids = []

def get_canonical(is_correct, score):
    """Return canonical correctness (True/False/None) and flag category."""
    ic_null = is_correct is None
    sc_null = score is None or (isinstance(score, float) and np.isnan(score))

    if ic_null and sc_null:
        return None, "both_null"
    elif (not ic_null) and (not sc_null):
        # Both present — check agreement
        if is_correct != (score == 1.0):
            return is_correct, "mismatch"
        return is_correct, "ok"
    else:
        # One is non-null — derive from whichever is available
        if not ic_null:
            return is_correct, "ok"
        else:
            return (score == 1.0), "ok"

for r in all_records:
    canonical, flag = get_canonical(r.get("is_correct"), r.get("score"))
    r["canonical_correct"] = canonical
    if flag == "both_null":
        both_null_conv_ids.append(r["conv_id"])
    elif flag == "mismatch":
        mismatch_conv_ids.append(r["conv_id"])

print(f"Total records: {len(all_records)}")
print(f"  Both null (is_correct=None, score=None/NaN): {len(both_null_conv_ids)}")
print(f"  Mismatch (both non-null but disagree):       {len(mismatch_conv_ids)}")
print(f"  Consistent / single-source:                  {len(all_records) - len(both_null_conv_ids) - len(mismatch_conv_ids)}")

if mismatch_conv_ids:
    print(f"\n⚠ Mismatched conv_ids (both non-null but disagree):")
    mismatch_records = [r for r in all_records if r["conv_id"] in set(mismatch_conv_ids)]
    display(pd.DataFrame([{
        "conv_id": r["conv_id"], "task": r["task"], "task_id": r["task_id"],
        "model": r["assistant_model"], "conv_type": r["conv_type"],
        "is_correct": r.get("is_correct"), "score": r.get("score"),
    } for r in mismatch_records]))

Total records: 6165
  Both null (is_correct=None, score=None/NaN): 0
  Mismatch (both non-null but disagree):       0
  Consistent / single-source:                  6165


In [4]:
# ── Step 2: Timestamp-based deduplication ──
# Extract timestamp from last trace entry for each record.
# If [task, model, conv_type, task_id] has multiple entries, keep the latest one.

def get_last_timestamp(record):
    """Extract timestamp from last trace entry (answer-evaluation or conversation-completed)."""
    trace = record.get("trace", [])
    for entry in reversed(trace):
        if "timestamp" in entry:
            return entry["timestamp"]
    return None

# Build flat records with timestamp, using canonical_correct from Step 1
flat_records = []
for r in all_records:
    canonical = r["canonical_correct"]
    flat_records.append({
        "conv_id": r["conv_id"],
        "task": r["task"],
        "task_id": r["task_id"],
        "model": r["assistant_model"],
        "conv_type": r["conv_type"],
        "canonical_correct": canonical,
        # Numeric score: True→1.0, False→0.0, None→NaN
        "score": 1.0 if canonical is True else (0.0 if canonical is False else np.nan),
        "timestamp": get_last_timestamp(r),
    })

df_all = pd.DataFrame(flat_records)
df_all["timestamp"] = pd.to_datetime(df_all["timestamp"])

print(f"Total records before dedup: {len(df_all)}")

# Count duplicates per key
dedup_key = ["task", "model", "conv_type", "task_id"]
dup_counts = df_all.groupby(dedup_key).size()
dups = dup_counts[dup_counts > 1]
print(f"Duplicate groups (same task/model/conv_type/task_id): {len(dups)}")
if len(dups) > 0:
    print(f"Max duplicates in a group: {dups.max()}")
    print(f"Total duplicate records to discard: {dups.sum() - len(dups)}")

# Keep the latest (most recent timestamp) for each group
df_all = df_all.sort_values("timestamp", ascending=False)
df_deduped = df_all.drop_duplicates(subset=dedup_key, keep="first").copy()
df_deduped = df_deduped.sort_values(dedup_key).reset_index(drop=True)

print(f"\nTotal records after dedup: {len(df_deduped)}")
print(f"  of which canonical_correct is None (NaN score): {df_deduped['score'].isna().sum()}")
print(f"\nRecords per (task, conv_type):")
display(df_deduped.groupby(["task", "conv_type"]).size().unstack(fill_value=0))

Total records before dedup: 6165
Duplicate groups (same task/model/conv_type/task_id): 462
Max duplicates in a group: 2
Total duplicate records to discard: 462

Total records after dedup: 5703
  of which canonical_correct is None (NaN score): 0

Records per (task, conv_type):


conv_type,full,sharded
task,,
actions,735,726
code,700,643
database,749,725
math,721,704


In [5]:
# ── Step 3: Aggregate summary DataFrame ──
# Columns: [task, model, conv_type, num_convs, total_score, average_score]

df_agg = (
    df_deduped
    .groupby(["task", "model", "conv_type"])
    .agg(
        num_convs=("score", "size"),
        total_score=("score", "sum"),
        average_score=("score", "mean"),
    )
    .reset_index()
    .sort_values(["task", "model", "conv_type"])
    .reset_index(drop=True)
)

print(f"Aggregate table: {len(df_agg)} rows")
display(df_agg)

Aggregate table: 56 rows


,task,model,conv_type,num_convs,total_score,average_score
0,actions,gpt-5,full,105,90.0,0.857143
1,actions,gpt-5,sharded,105,59.0,0.561905
2,actions,gpt-5-chat,full,105,102.0,0.971429
3,actions,gpt-5-chat,sharded,104,78.0,0.750000
4,actions,gpt-5-mini,full,105,85.0,0.809524
5,actions,gpt-5-mini,sharded,104,58.0,0.557692
6,actions,gpt-5.1,full,105,71.0,0.676190
7,actions,gpt-5.1,sharded,103,59.0,0.572816
8,actions,gpt-5.1-chat,full,105,96.0,0.914286
9,actions,gpt-5.1-chat,sharded,104,60.0,0.576923


In [6]:
# ── Step 4: Per-task_id pivot DataFrame ──
# Rows: unique task_id, Columns: {conv_type}_{model}
# Cells: 1.0 (correct), 0.0 (incorrect), or NaN (no entry)

# Create column name for each combination
df_deduped["col_name"] = df_deduped["conv_type"] + "_" + df_deduped["model"]

# Pivot: each row is a (task, task_id), columns are conv_type_model combos
df_pivot = df_deduped.pivot_table(
    index=["task", "task_id"],
    columns="col_name",
    values="score",
    aggfunc="first",  # should be unique after dedup
)

# Sort columns: group by conv_type then model
df_pivot = df_pivot.reindex(sorted(df_pivot.columns), axis=1)
df_pivot = df_pivot.reset_index()

print(f"Pivot table: {df_pivot.shape[0]} rows x {df_pivot.shape[1]} columns")
print(f"\nColumns: {list(df_pivot.columns)}")
print(f"\nNull counts per column:")
display(df_pivot.isnull().sum())
print()
display(df_pivot.head(20))

Pivot table: 415 rows x 16 columns

Columns: ['task', 'task_id', 'full_gpt-5', 'full_gpt-5-chat', 'full_gpt-5-mini', 'full_gpt-5.1', 'full_gpt-5.1-chat', 'full_gpt-5.2', 'full_gpt-5.2-chat', 'sharded_gpt-5', 'sharded_gpt-5-chat', 'sharded_gpt-5-mini', 'sharded_gpt-5.1', 'sharded_gpt-5.1-chat', 'sharded_gpt-5.2', 'sharded_gpt-5.2-chat']

Null counts per column:


col_name
task                     0
task_id                  0
full_gpt-5               0
full_gpt-5-chat          0
full_gpt-5-mini          0
full_gpt-5.1             0
full_gpt-5.1-chat        0
full_gpt-5.2             0
full_gpt-5.2-chat        0
sharded_gpt-5           64
sharded_gpt-5-chat       4
sharded_gpt-5-mini      11
sharded_gpt-5.1          8
sharded_gpt-5.1-chat     9
sharded_gpt-5.2          4
sharded_gpt-5.2-chat     7
dtype: int64

col_name,task,task_id,full_gpt-5,full_gpt-5-chat,full_gpt-5-mini,full_gpt-5.1,full_gpt-5.1-chat,full_gpt-5.2,full_gpt-5.2-chat,sharded_gpt-5,sharded_gpt-5-chat,sharded_gpt-5-mini,sharded_gpt-5.1,sharded_gpt-5.1-chat,sharded_gpt-5.2,sharded_gpt-5.2-chat
0,actions,sharded-BFCL/parallel_0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,actions,sharded-BFCL/parallel_1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,actions,sharded-BFCL/parallel_102,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0
3,actions,sharded-BFCL/parallel_103,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,actions,sharded-BFCL/parallel_105,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
5,actions,sharded-BFCL/parallel_107,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,actions,sharded-BFCL/parallel_109,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,actions,sharded-BFCL/parallel_111,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
8,actions,sharded-BFCL/parallel_113,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
9,actions,sharded-BFCL/parallel_116,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0


In [7]:
# ── Step 5: Degradation ranking (full→sharded) per task_id ──
# A model degrades on a problem iff full=True (1.0) AND sharded=False (0.0).
# If either is null for that model, skip it (don't count as degradation).

# Extract model names from pivot columns
models = sorted(set(
    c.replace("full_", "").replace("sharded_", "")
    for c in df_pivot.columns if c.startswith("full_") or c.startswith("sharded_")
))
print(f"Models: {models}")

# Count degradations per row
degradation_counts = []
for _, row in df_pivot.iterrows():
    count = 0
    eligible = 0
    for model in models:
        full_val = row.get(f"full_{model}")
        shard_val = row.get(f"sharded_{model}")
        # Skip if either is null
        if pd.isna(full_val) or pd.isna(shard_val):
            continue
        eligible += 1
        if full_val == 1.0 and shard_val == 0.0:
            count += 1
    degradation_counts.append({"n_degradations": count, "n_models_eligible": eligible})

df_degrad = df_pivot[["task", "task_id"]].copy()
df_degrad["n_degradations"] = [d["n_degradations"] for d in degradation_counts]
df_degrad["n_models_eligible"] = [d["n_models_eligible"] for d in degradation_counts]

# Sort descending by degradation count within each task
df_degrad = df_degrad.sort_values(["task", "n_degradations"], ascending=[True, False]).reset_index(drop=True)

print(f"\nDegradation distribution across all task_ids:")
display(df_degrad["n_degradations"].describe())

for task in sorted(df_degrad["task"].unique()):
    subset = df_degrad[df_degrad["task"] == task]
    print(f"\n{'='*60}")
    print(f"Task: {task}  (top 20 most degraded)")
    print(f"{'='*60}")
    display(subset.head(20))

Models: ['gpt-5', 'gpt-5-chat', 'gpt-5-mini', 'gpt-5.1', 'gpt-5.1-chat', 'gpt-5.2', 'gpt-5.2-chat']

Degradation distribution across all task_ids:


count    415.000000
mean       2.272289
std        2.028916
min        0.000000
25%        1.000000
50%        2.000000
75%        3.500000
max        7.000000
Name: n_degradations, dtype: float64


Task: actions  (top 20 most degraded)


col_name,task,task_id,n_degradations,n_models_eligible
0,actions,sharded-BFCL/parallel_0,7,7
1,actions,sharded-BFCL/parallel_198,7,7
2,actions,sharded-BFCL/parallel_33,7,7
3,actions,sharded-BFCL/parallel_1,6,7
4,actions,sharded-BFCL/parallel_103,6,7
5,actions,sharded-BFCL/parallel_137,6,7
6,actions,sharded-BFCL/parallel_177,6,7
7,actions,sharded-BFCL/parallel_4,6,7
8,actions,sharded-BFCL/parallel_49,6,7
9,actions,sharded-BFCL/parallel_80,6,7



Task: code  (top 20 most degraded)


col_name,task,task_id,n_degradations,n_models_eligible
105,code,sharded-HumanEval/62,6,6
106,code,sharded-livecodebench/2881,6,7
107,code,sharded-HumanEval/141,5,6
108,code,sharded-HumanEval/113,4,7
109,code,sharded-HumanEval/128,4,6
110,code,sharded-HumanEval/138,4,7
111,code,sharded-HumanEval/159,4,6
112,code,sharded-livecodebench/2825,4,5
113,code,sharded-HumanEval/105,3,7
114,code,sharded-HumanEval/153,3,6



Task: database  (top 20 most degraded)


col_name,task,task_id,n_degradations,n_models_eligible
205,database,sharded-spider-val-14-medium,7,7
206,database,sharded-spider-val-389-medium,7,7
207,database,sharded-spider-val-457-medium,7,7
208,database,sharded-spider-val-531-medium,7,7
209,database,sharded-spider-val-555-medium,7,7
210,database,sharded-spider-val-689-medium,7,7
211,database,sharded-spider-val-699-medium,7,7
212,database,sharded-spider-val-932-medium,7,7
213,database,sharded-spider-val-942-medium,7,7
214,database,sharded-spider-val-946-medium,7,7



Task: math  (top 20 most degraded)


col_name,task,task_id,n_degradations,n_models_eligible
312,math,sharded-GSM8K/315,7,7
313,math,sharded-GSM8K/855,7,7
314,math,sharded-GSM8K/1124,6,7
315,math,sharded-GSM8K/808,6,7
316,math,sharded-GSM8K/1190,5,6
317,math,sharded-GSM8K/1287,5,7
318,math,sharded-GSM8K/144,5,7
319,math,sharded-GSM8K/234,5,7
320,math,sharded-GSM8K/543,5,7
321,math,sharded-GSM8K/1066,4,7


In [8]:
# next we want to get the top 40 for each task
# we should compare task IDs in `data/t30d.json` because these should match the top 30
# then we should make a new data file with the 10 examples per task that are #30-40
# then we should add the full spec QA

In [9]:
# load in the t30d data
t30d_data = read_json(DATA_DIR / "t30d.json")
t30d_ids = set()
for item in t30d_data:
    t30d_ids.add(item["task_id"])

In [10]:
# get top 40 for each task
top40_df = {}
top30_to_40_ids = []
for task in ["math", "code", "actions", "database"]:
    subset = df_degrad[df_degrad["task"] == task].head(40)
    top40_df[task] = subset
    for i, (_, row) in enumerate(subset.iterrows()):
        if i >= 30:
            top30_to_40_ids.append(row["task_id"])
            continue
        task_id = row["task_id"]
        if task_id not in t30d_ids:
            print(f"⚠ Task ID {task_id} in top 30 of {task} not found in t30d.json")

# we expect 10 task_ids per task in the 30-40 range, so 40 total across 4 tasks
print(f"\nTotal task_ids in top 30-40 across all tasks: {len(top30_to_40_ids)}")


Total task_ids in top 30-40 across all tasks: 40


In [11]:
# load full data (LiC set)
full_lic_path = DATA_DIR / "sharded_instructions_600.json"
full_lic_data = read_json(full_lic_path)
full_lic_dict = {item["task_id"]: item for item in full_lic_data}

In [12]:
# get top 10 subset
top30_to_40_list = []
for task_id in top30_to_40_ids:
    if task_id not in full_lic_dict:
        print(f"⚠ Task ID {task_id} not found in full LiC data")
        continue
    item = full_lic_dict[task_id]
    top30_to_40_list.append(item)

In [13]:
# add full spec QA

from datasets import load_dataset

task_subset = ["math", "code", "actions", "database"]
he_dataset = load_dataset("openai/openai_humaneval")

def add_full_spec_qa(data: list) -> None:
    for item in data:
        task = item["task"]
        if task not in task_subset:
            continue

        if task == "math":
            item["full_spec_q"] = item["question"]
            item["ground_truth_a"] = item["answer"]
        elif task == "code":
            # task id options:
            # - sharded-HumanEval/{number}
            # - sharded-livecodebench/{number}
            if item["task_id"].startswith("sharded-HumanEval/"):
                item["full_spec_q"] = item.get("prompt", None)
                number = int(item["task_id"].split("/")[1])
                item["ground_truth_a"] = he_dataset["test"][number]["canonical_solution"]
            else:
                item["full_spec_q"] = item.get("question_content", None)
                # NOTE [2026.01.27]
                # - we can use execution-v2 dataset to get ground truth answers
                # - there's not a unique answer per question, the execution-v2 dataset contains multiple per question_id
                # - we can just pick any one of them (e.g. first or last)
                # - deferring for now
                item["ground_truth_a"] = None
                if item["full_spec_q"] is None:
                    print(
                        f"livecodebench task id: {item['task_id']}, no full spec question available"
                    )
        elif task == "actions":
            item["full_spec_q"] = item["fully_specified_question"][0][0]["content"]
            item["ground_truth_a"] = item["reference_answer"]
        elif task == "database":
            item["full_spec_q"] = item["fully_specified_question"]
            item["ground_truth_a"] = item["reference_sql"]
        else:
            print(f"skipping {item['task_id']}")
            item["full_spec_q"] = None
            item["ground_truth_a"] = None

/Users/matthewho/miniconda3/envs/collabmem/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 0a41a237-4462-43b2-b1f1-0fe530b75cd3)')' thrown while requesting HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/dataset_infos.json
Retrying in 1s [Retry 1/5].


In [14]:
add_full_spec_qa(top30_to_40_list)

In [15]:
# top 30 to 40
t3040d_path = DATA_DIR / "t3040d.json"
write_json(top30_to_40_list, t3040d_path)

wrote to /Users/matthewho/Documents/research/ctx_editor/data/t3040d.json


In [16]:
# ══════════════════════════════════════════════════════════════
# Analysis with different user simulator models
# ══════════════════════════════════════════════════════════════
# mini_user_logs -> user_model=gpt-5-mini, assistant_model=gpt-5-mini
# nano_user_logs -> user_model=gpt-5-nano, assistant_model=gpt-5-mini
# (sota_model_logs already loaded above with user_model=gpt-4o-mini)

USER_MODEL_LOG_DIRS = {
    "gpt-5-mini (user)": DATA_DIR / "imported_lic_logs/mini_user_logs",
    "gpt-5-nano (user)": DATA_DIR / "imported_lic_logs/nano_user_logs",
}

user_model_records = {}
for label, log_dir in USER_MODEL_LOG_DIRS.items():
    records = []
    for jsonl_file in sorted(log_dir.rglob("*.jsonl")):
        batch = read_jsonl(jsonl_file)
        records.extend(batch)
        print(f"  [{label}] {jsonl_file.relative_to(log_dir)}: {len(batch)} records")
    user_model_records[label] = records
    print(f"  Total for {label}: {len(records)}\n")

  [gpt-5-mini (user)] actions/full/full_actions_gpt-5-mini.jsonl: 105 records
  [gpt-5-mini (user)] actions/sharded/sharded_actions_gpt-5-mini.jsonl: 102 records
  [gpt-5-mini (user)] code/full/full_code_gpt-5-mini.jsonl: 100 records
  [gpt-5-mini (user)] code/sharded/sharded_code_gpt-5-mini.jsonl: 91 records
  [gpt-5-mini (user)] database/full/full_database_gpt-5-mini.jsonl: 107 records
  [gpt-5-mini (user)] database/sharded/sharded_database_gpt-5-mini.jsonl: 107 records
  [gpt-5-mini (user)] math/full/full_math_gpt-5-mini.jsonl: 103 records
  [gpt-5-mini (user)] math/sharded/sharded_math_gpt-5-mini.jsonl: 101 records
  Total for gpt-5-mini (user): 816

  [gpt-5-nano (user)] actions/full/full_actions_gpt-5-mini.jsonl: 105 records
  [gpt-5-nano (user)] actions/sharded/sharded_actions_gpt-5-mini.jsonl: 102 records
  [gpt-5-nano (user)] code/full/full_code_gpt-5-mini.jsonl: 100 records
  [gpt-5-nano (user)] code/sharded/sharded_code_gpt-5-mini.jsonl: 94 records
  [gpt-5-nano (user)] data

In [17]:
# ── Canonicalize, dedup, aggregate for each user model dataset ──
# Reuse get_canonical and get_last_timestamp from above

def process_records(records, label):
    """Run steps 1-3 (canonicalize, dedup, aggregate) on a set of records."""
    # Step 1: Canonical correctness
    for r in records:
        canonical, flag = get_canonical(r.get("is_correct"), r.get("score"))
        r["canonical_correct"] = canonical

    # Step 2: Flatten + dedup
    flat = []
    for r in records:
        canonical = r["canonical_correct"]
        flat.append({
            "conv_id": r["conv_id"],
            "task": r["task"],
            "task_id": r["task_id"],
            "model": r["assistant_model"],
            "user_model": r.get("user_model", "unknown"),
            "conv_type": r["conv_type"],
            "canonical_correct": canonical,
            "score": 1.0 if canonical is True else (0.0 if canonical is False else np.nan),
            "timestamp": get_last_timestamp(r),
        })
    df = pd.DataFrame(flat)
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    dedup_key = ["task", "model", "conv_type", "task_id"]
    before = len(df)
    df = df.sort_values("timestamp", ascending=False)
    df = df.drop_duplicates(subset=dedup_key, keep="first").copy()
    df = df.sort_values(dedup_key).reset_index(drop=True)
    print(f"[{label}] {before} -> {len(df)} records after dedup")

    # Step 3: Aggregate
    df_agg = (
        df.groupby(["task", "model", "conv_type"])
        .agg(
            num_convs=("score", "size"),
            total_score=("score", "sum"),
            average_score=("score", "mean"),
        )
        .reset_index()
        .sort_values(["task", "model", "conv_type"])
        .reset_index(drop=True)
    )
    return df, df_agg


user_model_dfs = {}
user_model_aggs = {}
for label, records in user_model_records.items():
    df, df_agg = process_records(records, label)
    user_model_dfs[label] = df
    user_model_aggs[label] = df_agg

for label, df_agg in user_model_aggs.items():
    print(f"\n{'='*60}")
    print(f"Aggregate table for {label}")
    print(f"{'='*60}")
    display(df_agg)

[gpt-5-mini (user)] 816 -> 816 records after dedup
[gpt-5-nano (user)] 818 -> 818 records after dedup

Aggregate table for gpt-5-mini (user)


,task,model,conv_type,num_convs,total_score,average_score
0,actions,gpt-5-mini,full,105,86.0,0.819048
1,actions,gpt-5-mini,sharded,102,57.0,0.558824
2,code,gpt-5-mini,full,100,46.0,0.460000
3,code,gpt-5-mini,sharded,91,40.0,0.439560
4,database,gpt-5-mini,full,107,90.0,0.841121
5,database,gpt-5-mini,sharded,107,15.0,0.140187
6,math,gpt-5-mini,full,103,96.0,0.932039
7,math,gpt-5-mini,sharded,101,74.0,0.732673



Aggregate table for gpt-5-nano (user)


,task,model,conv_type,num_convs,total_score,average_score
0,actions,gpt-5-mini,full,105,84.0,0.800000
1,actions,gpt-5-mini,sharded,102,60.0,0.588235
2,code,gpt-5-mini,full,100,56.0,0.560000
3,code,gpt-5-mini,sharded,94,38.0,0.404255
4,database,gpt-5-mini,full,107,94.0,0.878505
5,database,gpt-5-mini,sharded,105,22.0,0.209524
6,math,gpt-5-mini,full,103,95.0,0.922330
7,math,gpt-5-mini,sharded,102,70.0,0.686275


In [18]:
# ── Cross-user-model comparison for gpt-5-mini (assistant) ──
# Compare the same assistant model (gpt-5-mini) across all three user models

# Extract gpt-5-mini rows from the sota analysis
sota_mini = df_agg[df_agg["model"] == "gpt-5-mini"].copy()
sota_mini["user_model"] = "gpt-4o-mini (user)"

# Combine with the new user model aggregates
comparison_rows = [sota_mini]
for label, agg_df in user_model_aggs.items():
    subset = agg_df.copy()
    subset["user_model"] = label
    comparison_rows.append(subset)

df_compare = pd.concat(comparison_rows, ignore_index=True)

# Pivot for easy comparison: rows = (task, conv_type), columns = user_model
df_compare_pivot = df_compare.pivot_table(
    index=["task", "conv_type"],
    columns="user_model",
    values="average_score",
)
# Reorder columns
col_order = ["gpt-4o-mini (user)", "gpt-5-nano (user)", "gpt-5-mini (user)"]
df_compare_pivot = df_compare_pivot[[c for c in col_order if c in df_compare_pivot.columns]]

print("Assistant model: gpt-5-mini — Average score by user simulator model")
print("=" * 70)
display(df_compare_pivot.round(4))

# Degradation (full→sharded) per user model
print("\n\nDegradation (full - sharded) per user model:")
print("=" * 70)
for task in sorted(df_compare_pivot.index.get_level_values("task").unique()):
    full_row = df_compare_pivot.loc[(task, "full")]
    shard_row = df_compare_pivot.loc[(task, "sharded")]
    degrad = full_row - shard_row
    print(f"\n{task}:")
    for col in df_compare_pivot.columns:
        print(f"  {col}: {full_row[col]:.3f} → {shard_row[col]:.3f}  (Δ = {degrad[col]:+.3f})")

Assistant model: gpt-5-mini — Average score by user simulator model


user_model          gpt-4o-mini (user)  gpt-5-nano (user)  gpt-5-mini (user)
task     conv_type                                                          
actions  full                   0.8000             0.8000             0.8190
         sharded                0.5882             0.5882             0.5588
code     full                   0.5600             0.5600             0.4600
         sharded                0.4043             0.4043             0.4396
database full                   0.8785             0.8785             0.8411
         sharded                0.2095             0.2095             0.1402
math     full                   0.9223             0.9223             0.9320
         sharded                0.6863             0.6863             0.7327



Degradation (full - sharded) per user model:

actions:
  gpt-4o-mini (user): 0.800 → 0.588  (Δ = +0.212)
  gpt-5-nano (user): 0.800 → 0.588  (Δ = +0.212)
  gpt-5-mini (user): 0.819 → 0.559  (Δ = +0.260)

code:
  gpt-4o-mini (user): 0.560 → 0.404  (Δ = +0.156)
  gpt-5-nano (user): 0.560 → 0.404  (Δ = +0.156)
  gpt-5-mini (user): 0.460 → 0.440  (Δ = +0.020)

database:
  gpt-4o-mini (user): 0.879 → 0.210  (Δ = +0.669)
  gpt-5-nano (user): 0.879 → 0.210  (Δ = +0.669)
  gpt-5-mini (user): 0.841 → 0.140  (Δ = +0.701)

math:
  gpt-4o-mini (user): 0.922 → 0.686  (Δ = +0.236)
  gpt-5-nano (user): 0.922 → 0.686  (Δ = +0.236)
  gpt-5-mini (user): 0.932 → 0.733  (Δ = +0.199)


In [19]:
# ── Per-task_id pivot + degradation ranking for each user model dataset ──

user_model_pivots = {}
user_model_degrads = {}

for label, df in user_model_dfs.items():
    df = df.copy()
    df["col_name"] = df["conv_type"] + "_" + df["model"]

    df_piv = df.pivot_table(
        index=["task", "task_id"],
        columns="col_name",
        values="score",
        aggfunc="first",
    )
    df_piv = df_piv.reindex(sorted(df_piv.columns), axis=1).reset_index()
    user_model_pivots[label] = df_piv

    # Degradation ranking
    models_here = sorted(set(
        c.replace("full_", "").replace("sharded_", "")
        for c in df_piv.columns if c.startswith("full_") or c.startswith("sharded_")
    ))

    degrad_rows = []
    for _, row in df_piv.iterrows():
        count = 0
        eligible = 0
        for model in models_here:
            full_val = row.get(f"full_{model}")
            shard_val = row.get(f"sharded_{model}")
            if pd.isna(full_val) or pd.isna(shard_val):
                continue
            eligible += 1
            if full_val == 1.0 and shard_val == 0.0:
                count += 1
        degrad_rows.append({"n_degradations": count, "n_models_eligible": eligible})

    df_deg = df_piv[["task", "task_id"]].copy()
    df_deg["n_degradations"] = [d["n_degradations"] for d in degrad_rows]
    df_deg["n_models_eligible"] = [d["n_models_eligible"] for d in degrad_rows]
    df_deg = df_deg.sort_values(["task", "n_degradations"], ascending=[True, False]).reset_index(drop=True)
    user_model_degrads[label] = df_deg

    print(f"\n{'='*60}")
    print(f"Degradation summary for {label}")
    print(f"{'='*60}")
    print(f"Models: {models_here}")
    display(df_deg["n_degradations"].describe())
    for task in sorted(df_deg["task"].unique()):
        subset = df_deg[df_deg["task"] == task]
        print(f"\n  {task}: top 10 most degraded")
        display(subset.head(10))


Degradation summary for gpt-5-mini (user)
Models: ['gpt-5-mini']


count    415.000000
mean       0.371084
std        0.483678
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: n_degradations, dtype: float64


  actions: top 10 most degraded


col_name,task,task_id,n_degradations,n_models_eligible
0,actions,sharded-BFCL/parallel_0,1,1
1,actions,sharded-BFCL/parallel_1,1,1
2,actions,sharded-BFCL/parallel_103,1,1
3,actions,sharded-BFCL/parallel_105,1,1
4,actions,sharded-BFCL/parallel_117,1,1
5,actions,sharded-BFCL/parallel_118,1,1
6,actions,sharded-BFCL/parallel_121,1,1
7,actions,sharded-BFCL/parallel_128,1,1
8,actions,sharded-BFCL/parallel_134,1,1
9,actions,sharded-BFCL/parallel_135,1,1



  code: top 10 most degraded


...

col_name,task,task_id,n_degradations,n_models_eligible
105,code,sharded-HumanEval/106,1,1
106,code,sharded-HumanEval/111,1,1
107,code,sharded-HumanEval/118,1,1
108,code,sharded-HumanEval/128,1,1
109,code,sharded-HumanEval/138,1,1
110,code,sharded-HumanEval/150,1,1
111,code,sharded-HumanEval/153,1,1
112,code,sharded-HumanEval/159,1,1
113,code,sharded-HumanEval/36,1,1
114,code,sharded-HumanEval/5,1,1



  database: top 10 most degraded


col_name,task,task_id,n_degradations,n_models_eligible
205,database,sharded-spider-val-1020-medium,1,1
206,database,sharded-spider-val-14-medium,1,1
207,database,sharded-spider-val-147-medium,1,1
208,database,sharded-spider-val-149-medium,1,1
209,database,sharded-spider-val-2-medium,1,1
210,database,sharded-spider-val-215-medium,1,1
211,database,sharded-spider-val-235-medium,1,1
212,database,sharded-spider-val-253-medium,1,1
213,database,sharded-spider-val-257-hard,1,1
214,database,sharded-spider-val-265-medium,1,1



  math: top 10 most degraded


col_name,task,task_id,n_degradations,n_models_eligible
312,math,sharded-GSM8K/1027,1,1
313,math,sharded-GSM8K/1066,1,1
314,math,sharded-GSM8K/1124,1,1
315,math,sharded-GSM8K/1132,1,1
316,math,sharded-GSM8K/1166,1,1
317,math,sharded-GSM8K/1190,1,1
318,math,sharded-GSM8K/1287,1,1
319,math,sharded-GSM8K/1303,1,1
320,math,sharded-GSM8K/143,1,1
321,math,sharded-GSM8K/144,1,1
